In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Calibration — Instructor Guide

Classroom guide for the CUDA-Q Academic **Calibration** resources: interactive bring-up visualizations and [NVIDIA Ising Calibration](https://www.nvidia.com/en-us/solutions/quantum-computing/ising/) for reading real calibration plots.

This is an **instructor resource**, not a numbered student lesson path. Numbered notebooks will land in this folder later.

→ Folder inventory and references: [README.md](README.md)  
→ Learning path card: [Calibration track](https://nvidia.github.io/cuda-q-academic/learningpath.html?track=track-calibration)  
→ Gallery: [Visualization Gallery — Calibration](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html)

## Learning goals

By the end of a short classroom session using these resources, students should be able to:

* Explain the **order** of common single-qubit calibration experiments (resonator → qubit spectroscopy → Rabi → coherence → readout → benchmarking → pulse shaping)
* Use a **visualization** to build one physical insight per experiment before looking at lab data
* Analyze a calibration plot with **Ising** while separating observations from inferences

**Prerequisites:** comfort with |0⟩ and |1⟩, basic gates, and reading a scientific plot. Hardware experience is helpful but not required.

## Suggested classroom flow

1. **Introduction** — your lecture, assigned reading, or classroom discussion of the calibration experiment, tailored to your student audience  
2. **Concept** — open the live experiment visualization from the [visualization gallery](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html); use the interactive controls until the main insight lands  
3. **Recognition** — show a matching real or [QCalEval](https://huggingface.co/datasets/nvidia/QCalEval) plot; ask what features match the widget  
4. **Setup** — [environment and API-key setup](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html) *or* the [NIM playground](https://build.nvidia.com/nvidia/ising-calibration-1-35b-a3b) (no key)  
5. **Analysis** — run Ising on the plot; critique observation vs inference  
6. **Next step** — discuss one bounded, testable calibration action

**Ways to use Ising:** first-pass plot reader, second opinion on a student's analysis, or a reasoning exemplar for structured calibration language.

## Running Ising

| Path | Needs | Best for |
|------|--------|----------|
| [NIM playground](https://build.nvidia.com/nvidia/ising-calibration-1-35b-a3b) | Browser only | Lecture demos, zero-setup homework |
| [Environment and API-key setup](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html) | Free NVIDIA API key + local Python/Jupyter | Workshop prep: venv → kernel → `NVIDIA_API_KEY` |
| Cells below | API key + `requests` / `ipywidgets` | Colab or Jupyter upload-and-run |

**API keys:** create or manage keys via [NVIDIA Build](https://build.nvidia.com/). Never hard-code, print, save, or commit a key.

**Sample plots:** a few PNGs from [QCalEval](https://huggingface.co/datasets/nvidia/QCalEval) are enough — no full dataset download required.

Typical structured responses cover experiment description, conclusion, significance, fit quality, parameter extraction, and success classification. Always separate **direct observations** from **inferences**, and treat model output as a starting point for expert review.

## Notebook API (upload and run)

Install dependencies if needed, then run the interface cell. Edit the prompt, upload one PNG/JPEG, paste the key into the password field, and select **Run Ising**.

Model used: `nvidia/ising-calibration-1.5-31b` via the NVIDIA integrate API.

In [ ]:
# Install if needed
%pip install -q requests ipywidgets

In [ ]:
import base64
import requests
import ipywidgets as widgets
from IPython.display import display

# Auto-detect environment (Colab FileUpload layout differs from JupyterLab)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROMPT = """Experiment context:
This is a quantum-calibration experiment plot.

Please:
1. Describe the axes and important visible features.
2. Assess the raw-data quality and any displayed fit separately.
3. Extract only parameters supported by the plot, including units.
4. Recommend one bounded, testable next calibration step.
5. Return null rather than guessing when information is unavailable.

Distinguish direct observations from inferences."""


def run_api(prompt, api_key, mime_type, image_b64):
    payload = {
        "model": "nvidia/ising-calibration-1.5-31b",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime_type};base64,{image_b64}"},
                },
            ],
        }],
        "temperature": 0.2,
        "max_tokens": 8192,
        "stream": False,
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=180,
    )
    if not response.ok:
        print(f"HTTP {response.status_code}")
        print(response.text[:2000])
        response.raise_for_status()
    print(response.json()["choices"][0]["message"]["content"])


prompt_box = widgets.Textarea(
    value=PROMPT,
    layout=widgets.Layout(width="100%", height="200px"),
)
uploader = widgets.FileUpload(accept="image/*", multiple=False)
api_key_box = widgets.Password(placeholder="Enter your NVIDIA API key")
run_button = widgets.Button(description="Run Ising", button_style="success")
out = widgets.Output()


def on_click(b):
    with out:
        out.clear_output()
        if not uploader.value:
            print("Please upload an image first.")
            return
        if not api_key_box.value:
            print("Please enter your API key.")
            return
        print("Calling Ising...")
        if IN_COLAB:
            uploaded_file = next(iter(uploader.value.values()))
            mime_type = uploaded_file["metadata"]["type"] or "image/png"
            image_b64 = base64.b64encode(uploaded_file["content"]).decode("utf-8")
        else:
            uploaded_file = uploader.value[0]
            mime_type = uploaded_file["type"] or "image/png"
            image_b64 = base64.b64encode(uploaded_file["content"]).decode("utf-8")
        run_api(prompt_box.value, api_key_box.value, mime_type, image_b64)


run_button.on_click(on_click)

display(
    widgets.Label("1. Edit your prompt:"),
    prompt_box,
    widgets.Label("2. Upload image:"),
    uploader,
    widgets.Label("3. Enter API key:"),
    api_key_box,
    run_button,
    out,
)

## Prompting and review

Ask for:

* Axes, units, and visible features
* Raw-data quality and displayed-fit quality as separate assessments
* Extracted values only when labels or plotted features support them
* Direct observations separated from physical inferences
* One bounded, testable next calibration step
* `null` for unavailable information

Verify every numerical value against the plot before acting on the response.

### Troubleshooting

* **401 / 403:** confirm the API key is valid and has not expired
* **Model not found:** check the current identifier in [NVIDIA API docs](https://docs.api.nvidia.com/nim/reference/nvidia-ising-calibration-1-35b-a3b) or NVIDIA Build
* **Widget does not render:** rerun the install cell, restart the kernel, rerun the interface; for local kernels use the [setup guide](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html)
* **Upload fails:** use a single RGB PNG or JPEG; reduce very large images
* **Timeout:** retry once, then try a smaller image or the browser playground
* **Weak analysis:** name the experiment family and expected quantity, but do not disclose the answer you want inferred

Full references are in [README.md](README.md).